# 03 COMSOL simulation: flow field, particle trajectories and loads

Evaluation of the COMSOL half model at a total flow rate of 300 µL/min (Methods, Sections `sec:methods_comsol` and
`sec:methods_particles`). All functions are in `comsol_analysis.py`.

**Inputs** (folder `comsol/`, or the environment variable `COMSOL_DIR`)

| File | Content |
|---|---|
| `axis_300.txt` | cut line on the channel axis at mid-height: x, y, z (mm), p (Pa), spf.U (m/s) |
| `velocity_cross_300.txt` | cut line across the channel at the imaging position, mid-height: x, y, z (mm), spf.U (m/s), spf.sr (1/s) |
| `path_300_d15_v3.csv.gz`, `path_300_d20_v3.csv.gz` | particle tracks, converted from the COMSOL export with `comsol_convert.py` |

and `results/measured/measured_summary.csv` from `02_measured_summary.ipynb`.

**Coordinates.** $s$ is the distance downstream of the entrance of the focusing channel. The entrance is the position of
the steepest increase of the velocity on the axis ($y=-0.7455$ mm in the COMSOL coordinates). The channel axis is the
symmetry plane $x=-14.5$ mm. The assumed imaging position is $s=1.5$ mm (Methods).

**Output:** `results/comsol/*.csv`

In [ ]:
import pandas as pd
from IPython.display import display

import comsol_analysis as ca

pd.set_option('display.precision', 3)
pd.set_option('display.width', 200)
print('COMSOL files from:', ca.COMSOL_DIR.resolve())

## 1 Flow on the channel axis (Sections `sec:res-flow` and `sec:res-comsol`)

* `U_dev_*`: median, minimum and maximum of the axis velocity between $s=1.0$ and 2.3 mm (developed part).
* `s90_mm`, `s99_mm`: first position downstream of the entrance where the axis velocity reaches 90 % and 99 % of
  `U_dev_median`. The columns `*_analytical_mm` use the analytical centreline velocity instead.
* `dpdx_comsol_kPa_mm`: slope of a straight line fitted to $p(s)$ between $s=1.0$ and 2.3 mm.
* `dp_entrance_outlet_bar`: pressure at the entrance relative to the outlet, over `length_entrance_outlet_mm`.

In [ ]:
axis = ca.read_axis()
display(pd.Series(ca.axis_summary(axis)).to_frame('value'))

## 2 Velocity across the channel at the imaging position

The simulated velocity in the mid-plane is compared with Eq. `poiseuille_rectangular_velocity` at 300 µL/min.
`y_agree_um` is the largest distance from the axis up to which all points agree within 0.5 %. `diff_beyond_*` is the
range of the difference (COMSOL minus model) closer to the wall.

In [ ]:
cross_numbers, cross = ca.cross_summary(ca.read_cross())
display(pd.Series(cross_numbers).to_frame('value'))

## 3 Particle trajectories (Sections `sec:res-particles` and `sec:res-position`)

Twenty particles per diameter were released at the sample inlet. A particle is **passed** if it reaches the outlet and
**valid** if, in addition, its centre starts at least one radius away from the floor and the ceiling (Methods).
The table gives minimum, median and maximum over the valid particles: start height, height at the entrance and at
the imaging position, distance from the axis and velocity at the imaging position, relative velocity difference to the
fluid at the particle centre (`slip`, also over all passed particles), the peak extensional stress
$\sigma_\mathrm{e}=2\eta\,|\partial v/\partial y|$ within $|s|<0.3$ mm, and the time at which the particles leave
the model.

In [ ]:
data = ca.load_all_paths()
particles = ca.particle_summary(data)
display(particles.T)

## 4 Forces on the particles

Median and maximum magnitude of the total, drag and lift force over all samples of the valid particles in the focusing
channel ($0.3<s<2.9$ mm).

In [ ]:
display(ca.force_summary(data))

## 5 Particle acceleration at the channel entrance (Section `sec:res-loads`)

Per particle: $v_\mathrm{up}$ = median velocity for $-1.0<s<-0.5$ mm, $v_\mathrm{dn}$ = median velocity for
$1.0<s<1.5$ mm. The velocity is smoothed with a running median over 5 samples and made monotonic. $t_{10}$ and $t_{90}$
are the times at which it reaches 10 % and 90 % of the increase. Mean acceleration
$a_\mathrm{mean}=0.8\,(v_\mathrm{dn}-v_\mathrm{up})/(t_{90}-t_{10})$, maximum acceleration over 50 µs, and force
$F=ma$ with $m=\rho_\mathrm{p}\pi d^3/6$, $\rho_\mathrm{p}=1050$ kg/m³, $d=15$ µm.

The Results state that the 14 valid particles were evaluated (`selection='valid'`). The first version of this
evaluation used all particles that reach $s=1.5$ mm (`selection='reached_imaging'`). Both are shown so that the numbers
in the text can be checked against the right set.

In [ ]:
for selection in ['valid', 'reached_imaging']:
    acc = ca.particle_acceleration(*data[15], d_sim_um=15.0, selection=selection)
    print(f'{selection}: {len(acc)} particles, mass {acc.attrs["mass_kg"]:.3g} kg')
    display(acc.drop(columns='particle').agg(['min', 'median', 'max']).T)

## 6 Viscous stress on the cells along the simulated paths

The 15 µm trajectories are used for both cell lines, because this simulated diameter is closest to the measured median
diameters at 300 µL/min. Along each valid path, the shear stress at the outer cell edge is computed with Eq.
`shear_stress_rectangular` at $|x|+d/2$, where $x$ is the lateral offset of the particle from the axis and $d$ the
measured median diameter of the cell line. The table gives, per cell line, the median over the particles of the edge
stress between $s=0.3$ mm and the imaging position, and of the peak extensional stress, with the interquartile range
(`q25`, `q75`) and the range (`min`, `max`) over the particles.

In [ ]:
flow = ca.load_measured()
used = ca.used_diameters(data)
print('measured diameter at 300 µL/min:', {c: p['d_um'] for c, p in ca.CELLS.items()}, '| simulated diameter used:', used)
paths = ca.table_paths(data, used)
display(ca.path_stress_summary(paths))

## 7 Shear stress for a centred cell and passage time

For each condition, the shear stress at the edge of a centred cell with the measured median diameter of that condition
(Eq. `shear_stress_rectangular`), and the passage time from the entrance to the assumed imaging position,
$1.5\ \mathrm{mm}/v$, with the measured median velocity $v$ (BF and Ph2 pooled).

In [ ]:
display(ca.table_channel(flow))

## 8 Save all tables

In [ ]:
res = ca.run(save=True)
print('written to', ca.RESULTS_DIR.resolve())